# Stage 2: Trajectory Collection (Colab A100)

Runs `src/agent_loop.py` on A100 with bf16 precision.

**Setup:** Upload the repo as a zip or clone from git, then run cells in order.

In [ ]:
# Cell 1: Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name()}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Cell 2: Clone repo
!git clone https://github.com/agastyasridharan/bayes-and-confused.git /content/bayes-and-confused

import os
os.chdir('/content/bayes-and-confused')
!ls

In [ ]:
# Cell 3: Install dependencies
!pip install -q mp-api pymatgen pyyaml transformers accelerate

In [ ]:
# Cell 4: Authenticate with Hugging Face (needed for Llama gated model)
from huggingface_hub import login
login()  # paste your HF token when prompted

In [ ]:
# Cell 5: Run trajectory collection
# Smoke test first (5 trajectories, ~2 min), then full run (1200 trajectories)
!python src/agent_loop.py --smoke-test
# Uncomment below for the full run after verifying smoke test output:
# !python src/agent_loop.py

In [ ]:
# Cell 5b: Run expert prompt variant (400 additional trajectories)
!python src/run_expert_prompt.py

In [ ]:
# Cell 6: Quick inspection of results
import json, random
random.seed(42)

with open('data/trajectories/all_trajectories.json') as f:
    trajs = json.load(f)

empty = [t for t in trajs if t['side'] == 'empty']
present = [t for t in trajs if t['side'] == 'data_present']
print(f'Total: {len(trajs)} ({len(empty)} empty, {len(present)} data-present)')

# Show 30 random empty-side trajectories for gate review
print('\n=== 30 RANDOM EMPTY-SIDE TRAJECTORIES ===')
for t in random.sample(empty, 30):
    print(f"\n--- [{t['trajectory_id']}] {t['system_prompt_variant']} | {t['formula']} | {t['property']} ---")
    print(f"Query: {t['user_query']}")
    print(f"Response: {t['assistant_response'][:500]}")
    print()

In [ ]:
# Cell 7: Download results (trajectories + activations)
from google.colab import files
!zip -qr trajectories.zip data/trajectories/
files.download('trajectories.zip')